Modeling: Time Delays
=====================

A measurable quantity of a point source is its time delay—the time it takes for light to travel from the source to the
observer for each multiple image of the point source (e.g., the quasar images). This is often expressed as the relative
time delay between each image and the image with the shortest time delay, which is often referred to as
the "reference image."

Time delays are commonly used in strong lensing analyses, for example to measure the Hubble constant, since
they are less affected by microlensing and can provide robust cosmological constraints.

This script describes how to perform point source lens modeling using the time delays of the point source dataset
as additional information on top of the positions of the point source, in case you are studying the Hubble constant
or another measureable quantity that uses time delays.

__Contents__

- **Model:** Compose the lens model fitted to the data.
- **Dataset:** Load and plot the strong lens dataset.
- **Point Solver:** We set up the `PointSolver`, which is used to compute the multiple images of the point source in.
- **Search:** Configure the non-linear search used to fit the model.
- **Analysis:** Create the Analysis object that defines how the model is fitted to the data.
- **Run Times:** Profiling the expected run time of the model-fit.
- **Result:** Overview of the results of the model-fit.
- **Cosmology:** Time Delay lenses allow the Hubble constant to be constrained, because the difference between the.

__Model__

This script fits a `PointDataset` data of a 'galaxy-scale' strong lens with a model where:

 - The lens galaxy's total mass distribution is an `Isothermal`.
 - The source `Galaxy` is a point source `Point`.

No external shear is included in the model, where it is for the `imaging` and `interferometer` examples. For a
quadruply imaged point source (8 data points) there is insufficient information to constrain an `Isothermal` plus
an external shear (9 parameters).

Where a shear is fitted, it is never attached to the lens galaxy: it is an `ExternalShear` held in an
`al.MassField`, which lives in the model's own `fields` slot beside `galaxies=` (and in the tracer's
`fields=` argument). The shear describes the tidal field of everything outside the system, so it is a property of
the system rather than of a galaxy. See `imaging/modeling.py` for the full description.

__Start Here Notebook__

If any code in this script is unclear, refer to the `point_source/start_here.ipynb` notebook.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Dataset__

Load the strong lens point-source dataset `simple`, which is the dataset we will use to perform point source 
lens modeling.

In [ ]:
dataset_name = "simple"
dataset_path = Path("dataset") / "point_source" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/point_source/simulator.py"],
        check=True,
    )

We now load the point source dataset we will fit using point source modeling. 

We load this data as a `PointDataset`, which contains the positions and time_delays of every point source. 

In [ ]:
dataset = al.from_json(
    file_path=dataset_path / "point_dataset_with_time_delays.json",
)

We can print this dictionary to see the dataset's `name`, `positions` and `time_delays` and noise-map values.

In [ ]:
print("Point Dataset Info:")
print(dataset.info)

We can also plot the positions of the `PointDataset`.

In [ ]:
aplt.subplot_point_dataset(dataset=dataset)

We next load an image of the dataset, because as described in the `point_source/start_here.ipynb` notebook, it is
useful for visualizing the point source dataset.

It is also passed to the analysis below, where it is used only for visualization and not in the likelihood:
`image_with_points.png` shows the data with the observed positions (before the fit) and `fit_image_with_points.png`
adds the model positions and critical curves.

In [ ]:
data = al.Array2D.from_fits(file_path=dataset_path / "data.fits", pixel_scales=0.05)


aplt.plot_array(array=data, title="")

__Point Solver__

We set up the `PointSolver`, which is used to compute the multiple images of the point source in the image-plane.

There are no special settings or inputs for the fitting of time_delays, therefore the `PointSolver` is set up in the same way
as in the `point_source/start_here.ipynb` notebook.

In [ ]:
grid = al.Grid2D.uniform(
    shape_native=(100, 100),
    pixel_scales=0.2,  # <- The pixel-scale describes the conversion from pixel units to arc-seconds.
)

solver = al.PointSolver.for_grid(
    grid=grid,
    pixel_scale_precision=0.001,
    magnification_threshold=0.1,
)

__Model__

We compose a lens model where:

 - The lens galaxy's total mass distribution is an `Isothermal` [5 parameters].
 - The source galaxy is a `PointSolved` point source [0 parameters — its source-plane centre is solved
   analytically at every likelihood evaluation].

The number of free parameters and therefore the dimensionality of non-linear parameter space is N=5.

Name pairing is used as before to pair the `PointDataset` to the `Point` in the model, which is discussed below.

If you have fitted fluxes in the `fluxes` example, you will have seen that a `PointFlux` model component was used
which had the `flux` of the point source as an additional free parameter. For time delays, there is no special
model component or extra free parameters, because the time delays are a propety of the mass model.

In [ ]:
# Lens:

mass = af.Model(al.mp.Isothermal)

lens = af.Model(al.Galaxy, redshift=0.5, mass=al.mp.Isothermal)

# Source (`PointSolved`: the source-plane centre is solved analytically, not sampled; time delays
# are computed from the tracer at the observed positions, so they need no centre parameter either):

point_0 = af.Model(al.ps.PointSolved)

source = af.Model(al.Galaxy, redshift=1.0, point_0=point_0)

# Overall Lens Model:

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))

The `info` attribute shows the model in a readable format.

In [ ]:
print(model.info)

The same model can also be visualized as a figure, making its structure easier to understand at a glance.

The figure shows how the model is organized: which parameters belong to each component, and whether they are free, 
fixed, shared, linked by an expression, solved during the fit, or not configured. `model.info` provides the 
corresponding numerical details, including the prior assigned to each free parameter and the value of each fixed 
parameter.

In [ ]:
af.ModelPlotter(model).figure()

__Search__

The model is fitted to the data using the nested sampling algorithm Nautilus (see `start.here.py` for a 
full description).

In [ ]:
search = af.Nautilus(
    path_prefix=Path("point_source") / "features",
    name="time_delays",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see modeling examples
)

__Analysis__

Create the `AnalysisPoint` object defining how the via Nautilus the model is fitted to the data.

In [ ]:
analysis = al.AnalysisPoint(
    dataset=dataset,
    solver=solver,
    image=data,  # Visualization only (`image_with_points.png`, `fit_image_with_points.png`).
    # Positions use the default `FitPositionsImagePairAllSolved` (all-to-all pairing, solved centre).
)

__Analytic Reference Time (Solved)__

The default `FitTimeDelays` compares *relative* delays by subtracting the minimum delay from both
the data and the model (the "reference image" convention). An alternative is
`al.FitTimeDelaysSolved`, which instead solves a precision-weighted analytic reference time from
the data in closed form and analytically marginalizes over it — a smooth (and JAX-differentiable)
alternative to the min-subtraction. It is selected via `fit_time_delays_cls`, independently of the
positions fit (the model composition is unchanged):

In [ ]:
analysis_solved = al.AnalysisPoint(
    dataset=dataset,
    solver=solver,
    image=data,  # Visualization only (`image_with_points.png`, `fit_image_with_points.png`).
    fit_time_delays_cls=al.FitTimeDelaysSolved,  # Analytic reference time instead of min-subtraction.
)

__Run Times__

For the positions-only fit, the run time of the log likelihood function was ~0.01 seconds, which is fast

Evaluating the time delays does not increase this much, with a value of around ~0.01 seconds still expected.

Overall modeling run times should therefore be around 20 minutes on CPU, under 5 minutes on GPU.

__Model-Fit__

We begin the model-fit by passing the model and analysis object to the non-linear search (checkout the output folder
for on-the-fly visualization and results).

In [ ]:
result = search.fit(model=model, analysis=analysis)

__Result__

The `info` attribute shows the model in a readable format (if this does not display clearly on your screen refer to
`start_here.ipynb` for a description of how to fix this).

In [ ]:
print(result.info)


__Cosmology__

Time Delay lenses allow the Hubble constant to be constrained, because the difference between the geometric
time delay and the physical time delay is proportional to the Hubble constant.

We therefore create a Cosmology as a `Model` object in order to make the cosmological parameter Omega_m a free 
parameter.

In [ ]:
cosmology = af.Model(al.cosmo.FlatLambdaCDM)

By default, all parameters of a cosmology model are initialized as fixed values based on the Planck18 cosmology.

In order to make the Hubble constant, we override the default value of the Hubble constant with uniform prior.

In [ ]:
cosmology.H0 = af.UniformPrior(lower_limit=0.0, upper_limit=150.0)

# Overall Lens Model:

lens.mass.centre.centre_0 = 0.0
lens.mass.centre.centre_1 = 0.0
lens.mass.einstein_radius = 1.6

model = af.Collection(
    galaxies=af.Collection(lens=lens, source=source), cosmology=cosmology
)


The `info` attribute shows the model in a readable format

This confirms the model includes the Cosmology, which has the Hubble constant as a free parameter.

In [ ]:
print(model.info)

The cosmology enters the model as a component of its own beside `galaxies`: `FlatLambdaCDM`, with `H0` free and 
`Om0`, `Tcmb0`, `Neff`, `m_nu` and `Ob0` fixed to their Planck18 values. The lens's `centre` and 
`einstein_radius` were fixed just above, so this fit samples the lens ellipticity and the Hubble constant and 
nothing else.

In [ ]:
af.ModelPlotter(model).figure()

__Search__

The model is fitted to the data using the nested sampling algorithm Nautilus (see `start.here.py` for a 
full description).

In [ ]:
search = af.Nautilus(
    path_prefix=Path("point_source") / "features",
    name="time_delays_hubble_constant2",
    unique_tag=dataset_name,
    n_live=150,
    n_batch=50,
)

__Model-Fit__

We begin the model-fit by passing the model and analysis object to the non-linear search (checkout the output folder
for on-the-fly visualization and results).

In [ ]:
result = search.fit(model=model, analysis=analysis)

Checkout `autolens_workspace/*/guides/results` for a full description of analysing results.